In [63]:
import os
from ensurepip import bootstrap
import warnings
from gc import set_debug
from statistics import mean
from jupyter_builder.jupyterlab_semver import satisfies
from sklearn.impute import SimpleImputer
from optuna.trial import FixedTrial
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from sklearn.inspection import permutation_importance
import shap
from optuna.visualization import plot_param_importances, plot_optimization_history, plot_slice
import math
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import TargetEncoder


load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="adult")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM adults", engine)
df['workclass'] = df['workclass'].fillna('Private')
df['native_country'] = df['native_country'].fillna('United-States')
df = df.drop_duplicates()
top_6 = ['Prof-specialty',
    'Craft-repair',
    'Exec-managerial',
    'Adm-clerical',
    'Sales',
    'Other-service']
prods = df['occupation'].value_counts().loc[top_6]
prods = prods / prods.sum()
null_mask = df['occupation'].isna()
rng = np.random.default_rng(42)
df.loc[null_mask, 'occupation'] = rng.choice(top_6, size=null_mask.sum(), p=prods)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
df.shape

(32537, 15)

In [64]:
x = df.drop(columns=['income',  'sex', 'education'])
y = df['income']

In [65]:
x_tmp, x_test, y_tmp, y_test  = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)

In [66]:
cv_gen = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
t = ['workclass', 'marital_status', 'occupation', 'native_country', 'race', 'relationship']
prep = ColumnTransformer([('cat', TargetEncoder(cv=cv_gen, smooth='auto'), t)], remainder='passthrough')

In [ ]:
def unic(trial, model_name):
    if model_name == 'lr':
        params = {'C': trial.suggest_float('lr_C', 0.001, 1.0),
                  'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
                  'class_weight': trial.suggest_categorical('class_weight', [None, 'balanced']),
                  'solver': 'liblinear',
                  'max_iter': 1000,
                  'random_state': 42}
        model = Pipeline([('scaler', StandardScaler()),
                          ('clf', LogisticRegression(**params))])
    elif model_name == 'rf':
        